# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, not a dict

# Print basic metadata fields
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Authors: {getattr(metadata, 'author', [])}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** In Croissant, the concept of the record set, field, and columns all leverage `@id` values, which are used to programmatically reference and extract data.

In [ ]:
# Print all record sets and their details
record_set_dict = {}
print('Available Record Sets:')
for rs in getattr(metadata, 'recordSet', []):
    print(f"  - @id: {rs['@id']}")
    print(f"    name: {rs.get('name','')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"    Fields: {[field['@id'] for field in fields]}")
    record_set_dict[rs['@id']] = fields

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Record set and field `@id`s from above will be used.

Replace `<record_set_id>` with an actual record set `@id` identified from the overview section.

In [ ]:
# List all record set IDs
all_record_set_ids = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]
print(f"RecordSet IDs: {all_record_set_ids}")

# Prepare to load all record sets
dataframes = {}
for record_set_id in all_record_set_ids:
    print(f"Loading RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section can include removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Below is an example for the main RecordSet. Make sure to update field IDs to valid column names as shown above.**

In [ ]:
# Example: Choose a main record set and field for filtering
main_record_set_id = all_record_set_ids[0] if all_record_set_ids else None
df = dataframes[main_record_set_id] if main_record_set_id else pd.DataFrame()

# List columns to choose a numeric field:
print(f"Columns: {df.columns.tolist()}")

# Let's pick a likely numeric field, e.g., 'MW_kDa' if present. Replace as needed.
numeric_field = None
for col in df.columns:
    if 'mw' in col.lower() or 'abundance' in col.lower() or 'count' in col.lower() or 'quantity' in col.lower():
        numeric_field = col
        break
if not numeric_field:
    # Fallback: choose any column with numeric data
    for col in df.select_dtypes(include=['float', 'int']).columns:
        numeric_field = col
        break
if not numeric_field:
    print('No obvious numeric field found, please update the code with one!')

# Example threshold
threshold = 10

if numeric_field and numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (z-score):")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a categorical field (e.g., 'Sample', 'Description', etc.)
    group_field = None
    for col in df.columns:
        if col not in [numeric_field, norm_col] and df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        print(grouped_df.head())
else:
    print('No suitable numeric field available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualization for the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns and not df[numeric_field].isnull().all():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Dataset loaded from Croissant schema via `mlcroissant`.
- Inspected record sets and main fields.
- Demonstrated filtering, normalization, and grouping on numeric data.
- Provided basic data visualization.

`mlcroissant` enables FAIR, reproducible data extraction and transformation. For more detailed biological analysis, see dataset documentation or explore further!